In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import time

print("İP 2 Başladı: Eşleştirme Skoru (M) Hesaplanıyor...")

# -------------------------------------------------------------------------
# ADIM 1: VERİLERİ YÜKLE
# -------------------------------------------------------------------------
try:
    # Adayın vektörlerini yükle (numpy array)
    aday_vektorleri = np.load("IP1_Ciktisi_Aday_Vektorleri.npy")
    print(f"-> Aday verisi yüklendi. Beceri Sayısı: {aday_vektorleri.shape[0]}")

    # Proje verilerini yükle (pandas dataframe)
    df_projeler = pd.read_pickle("IP1_Ciktisi_Projeler_Vektorlu.pkl")
    print(f"-> {len(df_projeler)} adet proje yüklendi.")
    
except FileNotFoundError:
    print("HATA: İP 1 çıktı dosyaları bulunamadı. Lütfen önce İP 1'i tamamlayın.")
    raise

# -------------------------------------------------------------------------
# ADIM 2: SKORLAMA FONKSİYONU (KOSİNÜS BENZERLİĞİ)
# -------------------------------------------------------------------------
def calculate_matching_score(proje_vektorleri, aday_vektorleri):
    """
    Proje gereksinimleri ile aday becerileri arasındaki uyumu hesaplar.
    Yöntem: Her proje gereksinimi için adayın en iyi becerisini bul ve ortalamasını al.
    """
    # Eğer projenin beceri vektörü boşsa veya hatalıysa 0 döndür
    if proje_vektorleri is None or len(proje_vektorleri) == 0:
        return 0.0
    
    # Boyut kontrolü (Tek boyutlu array gelirse 2D yap)
    if len(proje_vektorleri.shape) == 1:
        proje_vektorleri = proje_vektorleri.reshape(1, -1)
        
    # 1. Benzerlik Matrisi Hesapla: (Proje Becerisi Sayısı x Aday Becerisi Sayısı)
    # Örn: Projenin 5 becerisi, Adayın 13 becerisi varsa -> (5x13) matris
    benzerlik_matrisi = cosine_similarity(proje_vektorleri, aday_vektorleri)
    
    # 2. Her proje gereksinimi için adayın EN İYİ (max) karşılayan becerisini bul
    # axis=1: Her satırın (proje becerisinin) maksimumunu al
    en_iyi_eslesmeler = benzerlik_matrisi.max(axis=1)
    
    # 3. Ortalama Al (Projenin ne kadar 'kapsandığı')
    # 0-1 arası değeri 0-100 puanına çevir
    skor = en_iyi_eslesmeler.mean() * 100
    
    return skor

# -------------------------------------------------------------------------
# ADIM 3: TÜM PROJELERİ PUANLA
# -------------------------------------------------------------------------
print("\nProjeler puanlanıyor (Bu işlem birkaç saniye sürebilir)...")
start_time = time.time()

# Her satır (proje) için skoru hesapla
df_projeler["M_Skoru"] = df_projeler["beceri_vektorleri"].apply(
    lambda p_vec: calculate_matching_score(p_vec, aday_vektorleri)
)

end_time = time.time()
print(f"-> Puanlama tamamlandı ({end_time - start_time:.2f} saniye).")

# -------------------------------------------------------------------------
# ADIM 4: SONUÇLARI SIRALA VE GÖSTER 
# -------------------------------------------------------------------------
# M_Skoru'na göre büyükten küçüğe sırala
df_sonuc = df_projeler.sort_values(by="M_Skoru", ascending=False)

print("\n🏆 ADAY İÇİN EN UYGUN İLK 10 PROJE (M-Skoru'na Göre) 🏆")
print("-" * 80)

# Sonuçları göster (Başlık yerine Link kullanıyoruz)
for i, row in df_sonuc.head(10).iterrows():
    # Linkten proje ismini tahmin etmeye çalışalım (URL'in son kısmı)
    proje_tahmini_isim = row['link'].split('/')[-1].replace('_en', '').replace('-', ' ').upper()
    
    print(f"{i+1}. [Skor: {row['M_Skoru']:.2f}] {proje_tahmini_isim}")
    print(f"   Link: {row['link']}")
    print(f"   Eşleşen Beceriler (Örnek): {row['cikarilan_beceriler'][:5]}...") 
    print("-" * 80)

# Sonuçları Kaydet (İP 5'te kullanılacak)
df_sonuc.to_pickle("IP2_Ciktisi_M_Skorlu_Projeler.pkl")
print("\nDosya kaydedildi: 'IP2_Ciktisi_M_Skorlu_Projeler.pkl'")

İP 2 Başladı: Eşleştirme Skoru (M) Hesaplanıyor...
-> Aday verisi yüklendi. Beceri Sayısı: 13
-> 952 adet proje yüklendi.

Projeler puanlanıyor (Bu işlem birkaç saniye sürebilir)...
-> Puanlama tamamlandı (0.23 saniye).

🏆 ADAY İÇİN EN UYGUN İLK 10 PROJE (M-Skoru'na Göre) 🏆
--------------------------------------------------------------------------------
149. [Skor: 99.87] 48828
   Link: https://youth.europa.eu/solidarity/opportunity/48828_en
   Eşleşen Beceriler (Örnek): ['social medium', 'lo', 'graphic design', 'e', 'persona']...
--------------------------------------------------------------------------------
348. [Skor: 99.86] 47877
   Link: https://youth.europa.eu/solidarity/opportunity/47877_en
   Eşleşen Beceriler (Örnek): ['english', 'social medium', 'com', 'usp', 'food']...
--------------------------------------------------------------------------------
204. [Skor: 99.86] 48664
   Link: https://youth.europa.eu/solidarity/opportunity/48664_en
   Eşleşen Beceriler (Örnek): ['commu